# Final VQA dataset checks

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
assert (REPO_ROOT / "pyproject.toml").exists(), "Could not locate the repository root."
sys.path.insert(0, str(REPO_ROOT))

# Edit this, or set BREXPERT_DATA_ROOT, if the data directory lives elsewhere.
DATA_ROOT = Path(os.environ.get("BREXPERT_DATA_ROOT", REPO_ROOT.parent / "data")).resolve()
print(f"DATA_ROOT = {DATA_ROOT}")

## Load the canonical VQA files

In [ ]:
import json

from utils.vqa_findings_templates import FIELD_SPECS

KNOWN_FIELD_IDS = {fs.field_id for fs in FIELD_SPECS} | {"lesion.presence"}

records = []
for variant_dir in sorted(DATA_ROOT.glob("*_vqa_split")):
    for split in ["train", "val", "test"]:
        path = variant_dir / f"vqa_{split}.jsonl"
        if not path.exists():
            continue
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                rec = json.loads(line)
                rec["_variant"] = variant_dir.name
                rec["_split_from_file"] = split
                records.append(rec)
print(f"{len(records)} dialogues loaded across {len({r['_variant'] for r in records})} variant(s)")
pd.DataFrame(records)[["_variant", "_split_from_file", "id", "source_id"]].head()

## Flatten conversation turns

In [ ]:
turn_rows = []
structure_issues = []
for rec in records:
    conv = rec.get("conversation", [])
    if len(conv) % 2 != 0:
        structure_issues.append({"dialogue_id": rec["id"], "issue": "odd_conversation_length", "length": len(conv)})
    for i, turn in enumerate(conv):
        expected_role = "user" if i % 2 == 0 else "assistant"
        if turn.get("role") != expected_role:
            structure_issues.append({"dialogue_id": rec["id"], "issue": "role_out_of_order", "position": i})
        if turn.get("role") == "assistant":
            turn_rows.append({
                "variant": rec["_variant"],
                "split": rec["_split_from_file"],
                "dialogue_id": rec["id"],
                "field_id": turn.get("field_id"),
                "is_negative": turn.get("is_negative", False),
                "negative_case": turn.get("negative_case"),
                "answer": turn.get("text"),
            })
turns_df = pd.DataFrame(turn_rows)
structure_issues_df = pd.DataFrame(structure_issues)
print(f"{len(turns_df)} assistant turns | {len(structure_issues_df)} structural issues")
structure_issues_df.head()

## Field id coverage

In [ ]:
unknown_fields = sorted(set(turns_df["field_id"].dropna()) - KNOWN_FIELD_IDS)
print(f"Unknown field ids: {unknown_fields}")
turns_df["field_id"].value_counts()

## Question-count consistency and zero-question dialogues

In [ ]:
dialogue_rows = []
for rec in records:
    assistant_turns = sum(1 for t in rec.get("conversation", []) if t.get("role") == "assistant")
    meta = rec.get("metadata", {})
    dialogue_rows.append({
        "variant": rec["_variant"],
        "split": rec["_split_from_file"],
        "dialogue_id": rec["id"],
        "actual_questions": assistant_turns,
        "metadata_question_count": meta.get("question_count"),
        "lesion_status": meta.get("lesion_status"),
    })
dialogues_df = pd.DataFrame(dialogue_rows)
mismatches = dialogues_df[dialogues_df["actual_questions"] != dialogues_df["metadata_question_count"]]
zero_question = dialogues_df[dialogues_df["actual_questions"] == 0]
print(f"question_count mismatches: {len(mismatches)} | zero-question dialogues: {len(zero_question)}")
dialogues_df.groupby(["variant", "split"])["actual_questions"].describe()

## Negative examples

In [ ]:
negatives_by_variant = turns_df.groupby("variant")["is_negative"].agg(["sum", "count", "mean"])
negatives_missing_case = turns_df[(turns_df["is_negative"]) & (turns_df["negative_case"].isna())]
print(f"negative turns missing a negative_case: {len(negatives_missing_case)}")
negatives_by_variant

## Lesion status distribution

In [ ]:
dialogues_df.groupby(["variant", "split", "lesion_status"]).size().unstack(fill_value=0)

## Orphan check

In [ ]:
split_dir = DATA_ROOT / "report_generation_split"
split_ids_by_split = {}
for split in ["train", "val", "test"]:
    path = split_dir / f"all-rg-{split}.csv"
    if path.exists():
        split_ids_by_split[split] = set(pd.read_csv(path, usecols=["id"])["id"].astype(str))

orphan_rows = []
for rec in records:
    valid_ids = split_ids_by_split.get(rec["_split_from_file"])
    if valid_ids is not None and str(rec["source_id"]) not in valid_ids:
        orphan_rows.append({"variant": rec["_variant"], "split": rec["_split_from_file"], "dialogue_id": rec["id"]})
print(f"orphan dialogues (source_id not found in split csv): {len(orphan_rows)}")
pd.DataFrame(orphan_rows).head()

## Summary

In [ ]:
print(f"Dialogues: {len(records)} | assistant turns: {len(turns_df)}")
print(f"Structural issues: {len(structure_issues_df)}")
print(f"Unknown field ids: {len(unknown_fields)}")
print(f"question_count mismatches: {len(mismatches)} | zero-question dialogues: {len(zero_question)}")
print(f"Negative turns missing a case label: {len(negatives_missing_case)}")
print(f"Orphan dialogues: {len(orphan_rows)}")